# Étape 02 — Classification des communes

Ce notebook montre, PAS À PAS, comment on obtient le cluster d'usage de CHAQUE commune française, même
celles qui n'ont jamais hébergé de capteur : `SOURCE_MODELE` (section 0) choisit entre les prédictions
de RÉFÉRENCE déjà connues (par défaut) et une forêt aléatoire fraîchement entraînée sur les capteurs
étiquetés de l'étape 01.

**Note sur cet environnement de démonstration** : ce notebook a besoin des sorties de
`00_transformation_des_donnees` (`commune_features.parquet`) et, si `SOURCE_MODELE = "modele"`, de
`01_clustering_des_usages` (`cluster_assignments.parquet`) - lancez leur `pipeline.py` respectif
d'abord si elles n'existent pas encore.

## 0. Configuration

`SOURCE_MODELE` est LE paramètre de ce notebook (voir le README de ce dossier pour la justification
complète : une forêt aléatoire n'est pas parfaitement déterministe d'un entraînement à l'autre).

In [1]:
import sys
from pathlib import Path

ICI = Path.cwd()
sys.path.insert(0, str(ICI))
RACINE = ICI.parents[1]

import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

DONNEES_VALIDES = RACINE / "data" / "donnees_valides"
FICHIER_COMMUNES = DONNEES_VALIDES / "commune_features.parquet"
FICHIER_CAPTEURS_ANNEES = DONNEES_VALIDES / "capteurs" / "sensor_years.parquet"
FICHIER_ASSIGNATION_CLUSTERS = DONNEES_VALIDES / "clustering" / "cluster_assignments.parquet"
FICHIER_REFERENCE = RACINE / "data" / "donnees_brutes" / "reference" / "commune_clusters_k4_reference.csv"
DOSSIER_SORTIE = DONNEES_VALIDES / "classification"
DOSSIER_MODELES = RACINE / "models" / "classification_des_communes"
FICHIER_MODELE = DOSSIER_MODELES / "foret_aleatoire.joblib"

# "reference" (par défaut) : prédictions déjà connues, vendorisées - résultats stables d'une exécution
# à l'autre. "modele" : entraîne (ou réutilise) une forêt aléatoire à la place - voir reference.py.
SOURCE_MODELE = "reference"

communes = pd.read_parquet(FICHIER_COMMUNES)
print(f"{len(communes):,} communes")

34,428 communes


## 1. Selon `SOURCE_MODELE` (section 0)

- `"reference"` : [`reference.py`](reference.py) - une jointure sur des prédictions déjà connues, voir
  son docstring pour pourquoi c'est le choix par défaut.
- `"modele"` : [`jeu_de_donnees.py`](jeu_de_donnees.py) construit le jeu d'entraînement à partir des
  capteurs étiquetés (étape 01), [`entrainement.py`](entrainement.py) ajuste une forêt aléatoire
  (réutilisée si déjà entraînée), [`prediction.py`](prediction.py) l'applique à toutes les communes.

In [2]:
if SOURCE_MODELE == "reference":
    from reference import predictions_reference

    predictions = predictions_reference(communes, FICHIER_REFERENCE)
    print(f"{len(predictions):,} communes classées (référence) - {int((predictions['extrapolee'] == 0).sum()):,} observées directement")

elif SOURCE_MODELE == "modele":
    from entrainement import ModeleClassification, ajuster_foret_aleatoire
    from jeu_de_donnees import clusters_rares, effectifs_par_cluster, jeu_entrainement
    from prediction import appliquer_modele_classification, preparer_classification_communes

    capteurs = pd.read_parquet(FICHIER_CAPTEURS_ANNEES)
    assignation_clusters = pd.read_parquet(FICHIER_ASSIGNATION_CLUSTERS)
    labels = assignation_clusters.set_index("id_site")["cluster"]
    jeu = jeu_entrainement(communes, capteurs, labels)
    print(f"{len(jeu.X):,} capteurs dans {jeu.groupes.nunique():,} communes, {jeu.X.shape[1]} variables")
    display(effectifs_par_cluster(jeu))
    print("clusters rares (< 10 communes) :", clusters_rares(jeu))

    # Priorité à un modèle déjà entraîné (même logique qu'à l'étape 01) : un modèle choisi à la main
    # ne doit jamais être silencieusement remplacé par un nouveau modèle aux paramètres par défaut.
    if FICHIER_MODELE.exists():
        print(f"\nmodèle déjà entraîné réutilisé : {FICHIER_MODELE}")
        modele = ModeleClassification.charger(FICHIER_MODELE)
        predictions = appliquer_modele_classification(modele, communes, capteurs, assignation_clusters)
    else:
        print("\naucun modèle sauvegardé : entraînement d'un nouveau modèle.")
        predictions, modele = preparer_classification_communes(communes, capteurs, assignation_clusters)
        DOSSIER_MODELES.mkdir(parents=True, exist_ok=True)
        modele.sauvegarder(FICHIER_MODELE)
    print(f"accuracy équilibrée hors-pli : {modele.accuracy_equilibree_hors_pli:.1%}")

else:
    raise ValueError(f"SOURCE_MODELE={SOURCE_MODELE!r} attendu parmi 'reference', 'modele'")

predictions["cluster_predit"].value_counts().sort_index()

34,428 communes classées (référence) - 1,007 observées directement


cluster_predit
0      907
1      402
2    20602
3    12517
Name: count, dtype: int64

## 2. Écrire les résultats

Comme le ferait `pipeline.py` - seulement si ce notebook est la source de vérité pour cette exécution
(sinon, relancez plutôt `pipeline.py`, qui a la même priorité référence/modèle).

In [3]:
DOSSIER_SORTIE.mkdir(parents=True, exist_ok=True)
chemin_predictions = DOSSIER_SORTIE / "commune_clusters.parquet"
predictions.to_parquet(chemin_predictions, index=False)
print("écrit :", chemin_predictions)

écrit : C:\Users\olivi\Documents\GitHub\LEVEL-Cyclist-Risk-Estimation\VV6\data\donnees_valides\classification\commune_clusters.parquet
